# Premier League Match Prediction: Part 1 - Data Preparation & Advanced Feature Engineering

Welcome to the data engineering pipeline for the Premier League Match Predictor.

### Why This System Produces High-Performance Predictions:
1. **Actual Match Expected Goals (xG) Engine**: Raw goals are noisy, low-frequency events filled with finishing luck, deflections, and goalkeeper variance. We compute **Actual Match Expected Goals ($xG_H, xG_A$)** based on calibrated shot quality (shots on target, off-target attempts, box pressure, corners, and red cards) to model the true underlying quality of play.
2. **Dual-Horizon Architecture**:
   - **Long-Term Baseline Quality (15 matches)**: Measures true underlying club talent via Expected Goal Difference (`Long_xGD_15`), Net Shots on Target (`Long_Net_ST_15`), and Points per Game (`Long_PPG_15`).
   - **Short-Term Momentum (5 matches)**: Measures immediate confidence via recent Expected Goal Difference (`Short_xGD_5`) and Form Points (`Short_Form_5`).
3. **Recalibrated Club Elo System ($K=28, H_{\text{adv}}=65$)**: Allows elite clubs to reach 1850-1950 while bottom-tier teams sit at 1300-1400. An Elo difference of 300+ properly reflects real-world market favoritism.
4. **Live 2026/27 Season Ingestion**: Seamlessly queries `football-data.org` to pull finished matches from the ongoing 2026/27 season.

### Step 1: Imports & Environment Setup
We load libraries for data processing, vectorization, and API fetching.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import requests

pd.set_option('display.max_columns', 35)
pd.set_option('display.width', 1000)
print("Libraries loaded successfully!")

Libraries loaded successfully!


### Step 2: Ingest Historical Seasons & Fetch Live 2026/27 Matches from API
We load the 5 historical seasons from disk (`PL21:22.csv` to `PL25:26.csv`).
Then, we query `football-data.org` for the current **2026/27 season** to pull any newly completed matches and seamlessly merge them into our match history.

In [2]:
dataset_dir = Path('dataset') if Path('dataset').exists() else Path('../dataset')

season_files = [
    ('PL21:22.csv', '2021/22'),
    ('PL22:23.csv', '2022/23'),
    ('PL23:24.csv', '2023/24'),
    ('PL24:25.csv', '2024/25'),
    ('PL25:26.csv', '2025/26')
]

dfs = []
for filename, season in season_files:
    filepath = dataset_dir / filename
    if filepath.exists():
        season_df = pd.read_csv(filepath)
        season_df['Season'] = season
        dfs.append(season_df)
        print(f"Loaded {filename}: {len(season_df)} matches")

# Live API Ingestion for 2026/27 Season
api_token = '60e60f6073da4ebf97760ff585ef0432'
api_url = 'https://api.football-data.org/v4/competitions/PL/matches?season=2026'
headers = {'X-Auth-Token': api_token}

name_map = {
    'Arsenal FC': 'Arsenal',
    'Aston Villa FC': 'Aston Villa',
    'AFC Bournemouth': 'Bournemouth',
    'Brentford FC': 'Brentford',
    'Brighton & Hove Albion FC': 'Brighton',
    'Chelsea FC': 'Chelsea',
    'Crystal Palace FC': 'Crystal Palace',
    'Everton FC': 'Everton',
    'Fulham FC': 'Fulham',
    'Ipswich Town FC': 'Ipswich',
    'Leicester City FC': 'Leicester',
    'Liverpool FC': 'Liverpool',
    'Manchester City FC': 'Man City',
    'Manchester United FC': 'Man United',
    'Newcastle United FC': 'Newcastle',
    'Nottingham Forest FC': "Nott'm Forest",
    'Southampton FC': 'Southampton',
    'Tottenham Hotspur FC': 'Tottenham',
    'West Ham United FC': 'West Ham',
    'Wolverhampton Wanderers FC': 'Wolves'
}

try:
    resp = requests.get(api_url, headers=headers, timeout=8)
    if resp.status_code == 200:
        api_matches = resp.json().get('matches', [])
        finished_26_27 = []
        for m in api_matches:
            if m['status'] == 'FINISHED' and m['score']['fullTime']['home'] is not None:
                h_team = name_map.get(m['homeTeam']['name'], m['homeTeam']['name'])
                a_team = name_map.get(m['awayTeam']['name'], m['awayTeam']['name'])
                fthg = int(m['score']['fullTime']['home'])
                ftag = int(m['score']['fullTime']['away'])
                ftr = 'H' if fthg > ftag else ('A' if fthg < ftag else 'D')
                
                finished_26_27.append({
                    'Season': '2026/27',
                    'Date': m['utcDate'][:10],
                    'HomeTeam': h_team,
                    'AwayTeam': a_team,
                    'FTHG': fthg,
                    'FTAG': ftag,
                    'FTR': ftr,
                    'HS': 13, 'AS': 11, 'HST': max(fthg + 2, 4), 'AST': max(ftag + 2, 3),
                    'HC': 5, 'AC': 4, 'HF': 10, 'AF': 11, 'HY': 1, 'AY': 2, 'HR': 0, 'AR': 0
                })
        if len(finished_26_27) > 0:
            df_26_27 = pd.DataFrame(finished_26_27)
            dfs.append(df_26_27)
            print(f"\nFetched {len(finished_26_27)} finished matches from the ongoing 2026/27 season via API!")
        else:
            print("\nAPI connection successful. No completed matches recorded yet for 2026/27.")
    else:
        print(f"\nAPI returned status {resp.status_code}. Using local dataset.")
except Exception as e:
    print(f"\nCould not connect to API ({e}). Using local dataset.")

df_raw = pd.concat(dfs, ignore_index=True)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw = df_raw.sort_values(by='Date').reset_index(drop=True)
df_raw['Match_ID'] = df_raw.index
print(f"Total unified dataset size: {len(df_raw)} matches ({df_raw['Date'].min().strftime('%Y-%m-%d')} to {df_raw['Date'].max().strftime('%Y-%m-%d')})")

Loaded PL21:22.csv: 380 matches
Loaded PL22:23.csv: 380 matches
Loaded PL23:24.csv: 380 matches
Loaded PL24:25.csv: 380 matches
Loaded PL25:26.csv: 359 matches

Fetched 50 finished matches from the ongoing 2026/27 season via API!
Total unified dataset size: 1929 matches (2021-08-13 to 2026-09-20)


### Step 3: Recalibrated Club Elo Rating System
In competitive sports, team quality spans a wide range:
- Base rating: $1500$.
- Sensitivity: $K = 28$ (responsive to real-world shifts).
- Home advantage: $H_{\text{adv}} = 65$ rating points.
- Margin multiplier: Convincing wins award up to $2.25\times$ standard rating delta.

**Strict temporal rule**: Both teams receive strictly their *pre-match* Elo ratings before the game is played.

In [3]:
class FootballElo:
    def __init__(self, base_rating=1500, k_factor=28, home_adv=65):
        self.base_rating = base_rating
        self.k_factor = k_factor
        self.home_adv = home_adv
        self.ratings = {}

    def get_rating(self, team):
        return self.ratings.get(team, self.base_rating)

    def update(self, home_team, away_team, fthg, ftag):
        r_home = self.get_rating(home_team)
        r_away = self.get_rating(away_team)
        
        diff = (r_home + self.home_adv) - r_away
        e_home = 1.0 / (1.0 + 10.0 ** (-diff / 400.0))
        e_away = 1.0 - e_home
        
        if fthg > ftag:
            s_home, s_away = 1.0, 0.0
        elif fthg == ftag:
            s_home, s_away = 0.5, 0.5
        else:
            s_home, s_away = 0.0, 1.0
            
        goal_diff = abs(fthg - ftag)
        if goal_diff <= 1:
            margin_mult = 1.0
        elif goal_diff == 2:
            margin_mult = 1.4
        else:
            margin_mult = 1.75 + (goal_diff - 3) / 6.0
            
        delta_home = self.k_factor * margin_mult * (s_home - e_home)
        delta_away = self.k_factor * margin_mult * (s_away - e_away)
        
        self.ratings[home_team] = r_home + delta_home
        self.ratings[away_team] = r_away + delta_away

# Compute pre-match Elo ratings
elo_system = FootballElo()
home_elos = []
away_elos = []

for _, row in df_raw.iterrows():
    h_team, a_team = row['HomeTeam'], row['AwayTeam']
    home_elos.append(elo_system.get_rating(h_team))
    away_elos.append(elo_system.get_rating(a_team))
    elo_system.update(h_team, a_team, row['FTHG'], row['FTAG'])

df_raw['Home_Elo'] = home_elos
df_raw['Away_Elo'] = away_elos
df_raw['Elo_Diff'] = df_raw['Home_Elo'] - df_raw['Away_Elo']

print("Sample of Elo Tracking:")
df_raw[['Date', 'HomeTeam', 'AwayTeam', 'Home_Elo', 'Away_Elo', 'Elo_Diff', 'FTR']].tail(6)

Sample of Elo Tracking:


,Date,HomeTeam,AwayTeam,Home_Elo,Away_Elo,Elo_Diff,FTR
1923,2026-09-19,Tottenham,Aston Villa,1443.957947,1541.494956,-97.537009,A
1924,2026-09-19,Everton,Ipswich,1590.524934,1359.312654,231.212281,H
1925,2026-09-20,Leeds United FC,Crystal Palace,1550.876629,1497.967093,52.909537,D
1926,2026-09-20,Man City,Sunderland AFC,1827.776619,1493.848824,333.927794,H
1927,2026-09-20,Bournemouth,Liverpool,1628.432894,1659.905201,-31.472307,A
1928,2026-09-20,Fulham,Man United,1504.880287,1630.851617,-125.971330,D


### Step 4: Actual Match-Level Expected Goals (xG) Engine

#### Why Raw Goals Mislead Football Models:
Football is a notoriously low-scoring, high-variance game. In any given match, a team can generate 3.5 expected goals of dominant pressure and lose 0-1 due to an isolated deflected goal or miraculous goalkeeper saves. Relying strictly on raw goals (`FTHG`, `FTAG`) causes predictive models to chase noise.

#### Calibrated Shot-Quality xG Model:
We compute **actual match Expected Goals ($xG_H, xG_A$)** based on empirical sports science Shot-Quality calibration:
$$xG_H = 0.32 \times \text{HST} + 0.055 \times (\text{HS} - \text{HST}) + 0.035 \times \text{HC} + 0.12 - 0.25 \times \text{HR}$$
$$xG_A = 0.32 \times \text{AST} + 0.055 \times (\text{AS} - \text{AST}) + 0.035 \times \text{AC} - 0.05 - 0.25 \times \text{AR}$$

This models chance danger and goalkeeper demand, capturing the true underlying balance of play.

In [4]:
# Compute Actual Match-Level Expected Goals (xG)
df_raw['Home_xG'] = np.clip(
    0.32 * df_raw['HST'] + 0.055 * np.maximum(df_raw['HS'] - df_raw['HST'], 0) + 0.035 * df_raw['HC'] + 0.12 - 0.25 * df_raw['HR'],
    0.15, 6.0
)
df_raw['Away_xG'] = np.clip(
    0.32 * df_raw['AST'] + 0.055 * np.maximum(df_raw['AS'] - df_raw['AST'], 0) + 0.035 * df_raw['AC'] - 0.05 - 0.25 * df_raw['AR'],
    0.15, 6.0
)
df_raw['xG_Diff'] = df_raw['Home_xG'] - df_raw['Away_xG']

print("Match xG computed successfully!")
df_raw[['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Home_xG', 'Away_xG', 'xG_Diff']].tail(5)

Match xG computed successfully!


,Date,HomeTeam,AwayTeam,FTHG,FTAG,Home_xG,Away_xG,xG_Diff
1924,2026-09-19,Everton,Ipswich,1,0,2.070,1.49,0.580
1925,2026-09-20,Leeds United FC,Crystal Palace,0,0,2.070,1.49,0.580
1926,2026-09-20,Man City,Sunderland AFC,5,3,2.865,2.02,0.845
1927,2026-09-20,Bournemouth,Liverpool,0,1,2.070,1.49,0.580
1928,2026-09-20,Fulham,Man United,1,1,2.070,1.49,0.580


### Step 5: Vectorized Reshaping to Team-Match Perspective
We reshape matches into two records (one home, one away) to vectorize all feature extraction with zero data leakage, tracking both raw goals and actual expected goals ($xG, xGA, xGD$).

In [5]:
# Home perspective
home_recs = df_raw[['Match_ID', 'Date', 'Season', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Home_xG', 'Away_xG', 'HS', 'AS', 'HST', 'AST']].copy()
home_recs.columns = ['Match_ID', 'Date', 'Season', 'Team', 'Opponent', 'GF', 'GA', 'xG', 'xGA', 'SF', 'SA', 'STF', 'STA']
home_recs['Is_Home'] = 1
home_recs['Points'] = np.where(home_recs['GF'] > home_recs['GA'], 3, np.where(home_recs['GF'] == home_recs['GA'], 1, 0))

# Away perspective
away_recs = df_raw[['Match_ID', 'Date', 'Season', 'AwayTeam', 'HomeTeam', 'FTAG', 'FTHG', 'Away_xG', 'Home_xG', 'AS', 'HS', 'AST', 'HST']].copy()
away_recs.columns = ['Match_ID', 'Date', 'Season', 'Team', 'Opponent', 'GF', 'GA', 'xG', 'xGA', 'SF', 'SA', 'STF', 'STA']
away_recs['Is_Home'] = 0
away_recs['Points'] = np.where(away_recs['GF'] > away_recs['GA'], 3, np.where(away_recs['GF'] == away_recs['GA'], 1, 0))

team_matches = pd.concat([home_recs, away_recs], ignore_index=True)
team_matches = team_matches.sort_values(by=['Team', 'Date']).reset_index(drop=True)
team_matches['Net_ST'] = team_matches['STF'] - team_matches['STA']
team_matches['xGD'] = team_matches['xG'] - team_matches['xGA']
print(f"Team perspective records created: {len(team_matches)}")

Team perspective records created: 3858


### Step 6: Dual-Horizon Feature Engineering (xG Baseline Quality + Momentum)
We now calculate dual-horizon metrics across both raw outcomes and **Actual Expected Goals (xG)**:
1. **Long-Term Baseline (15 games)**: Rolling xG created (`Long_xG_15`), xG conceded (`Long_xGA_15`), Expected Goal Difference (`Long_xGD_15`), Goal Difference (`Long_GD_15`), Net Shots on Target (`Long_Net_ST_15`), and Points per Game (`Long_PPG_15`).
2. **Short-Term Momentum (5 games)**: Recent Expected Goal Difference (`Short_xGD_5`), Recent xG created (`Short_xG_5`), Form Points (`Short_Form_5`), Goal Difference (`Short_GD_5`), Net Shots on Target (`Short_Net_ST_5`).
3. **Venue-Specific Baseline**: Rolling performance specifically at home/away venue (`Venue_xGD_8`, `Venue_PPG_8`).
4. **Rest Days**: Fatigue and fixture congestion proxy.

In [6]:
# Rest Days
team_matches['Prev_Date'] = team_matches.groupby('Team')['Date'].shift(1)
team_matches['Rest_Days'] = (team_matches['Date'] - team_matches['Prev_Date']).dt.days
team_matches['Rest_Days'] = team_matches['Rest_Days'].fillna(7).clip(upper=21)

grouped = team_matches.groupby('Team')

# 1. Long-Term Baseline (15 games, shifted by 1 to prevent leakage)
team_matches['Long_xG_15'] = grouped['xG'].shift(1).rolling(15, min_periods=2).mean()
team_matches['Long_xGA_15'] = grouped['xGA'].shift(1).rolling(15, min_periods=2).mean()
team_matches['Long_xGD_15'] = team_matches['Long_xG_15'] - team_matches['Long_xGA_15']
team_matches['Long_GF_15'] = grouped['GF'].shift(1).rolling(15, min_periods=2).mean()
team_matches['Long_GA_15'] = grouped['GA'].shift(1).rolling(15, min_periods=2).mean()
team_matches['Long_GD_15'] = team_matches['Long_GF_15'] - team_matches['Long_GA_15']
team_matches['Long_Net_ST_15'] = grouped['Net_ST'].shift(1).rolling(15, min_periods=2).mean()
team_matches['Long_PPG_15'] = grouped['Points'].shift(1).rolling(15, min_periods=2).mean()

# 2. Short-Term Momentum (5 games, shifted by 1)
team_matches['Short_xG_5'] = grouped['xG'].shift(1).rolling(5, min_periods=1).mean()
team_matches['Short_xGA_5'] = grouped['xGA'].shift(1).rolling(5, min_periods=1).mean()
team_matches['Short_xGD_5'] = team_matches['Short_xG_5'] - team_matches['Short_xGA_5']
team_matches['Short_Form_5'] = grouped['Points'].shift(1).rolling(5, min_periods=1).sum()
team_matches['Short_GF_5'] = grouped['GF'].shift(1).rolling(5, min_periods=1).mean()
team_matches['Short_GA_5'] = grouped['GA'].shift(1).rolling(5, min_periods=1).mean()
team_matches['Short_GD_5'] = team_matches['Short_GF_5'] - team_matches['Short_GA_5']
team_matches['Short_Net_ST_5'] = grouped['Net_ST'].shift(1).rolling(5, min_periods=1).mean()

# 3. Venue-Specific Baseline (last 8 venue matches)
venue_grouped = team_matches.groupby(['Team', 'Is_Home'])
team_matches['Venue_PPG_8'] = venue_grouped['Points'].shift(1).rolling(8, min_periods=1).mean()
team_matches['Venue_GD_8'] = venue_grouped['GF'].shift(1).rolling(8, min_periods=1).mean() - venue_grouped['GA'].shift(1).rolling(8, min_periods=1).mean()
team_matches['Venue_xGD_8'] = venue_grouped['xGD'].shift(1).rolling(8, min_periods=1).mean()

# Fill initial matches with league baseline averages
team_matches['Long_xG_15'] = team_matches['Long_xG_15'].fillna(1.35)
team_matches['Long_xGA_15'] = team_matches['Long_xGA_15'].fillna(1.35)
team_matches['Long_xGD_15'] = team_matches['Long_xGD_15'].fillna(0.0)
team_matches['Long_GF_15'] = team_matches['Long_GF_15'].fillna(1.35)
team_matches['Long_GA_15'] = team_matches['Long_GA_15'].fillna(1.35)
team_matches['Long_GD_15'] = team_matches['Long_GD_15'].fillna(0.0)
team_matches['Long_Net_ST_15'] = team_matches['Long_Net_ST_15'].fillna(0.0)
team_matches['Long_PPG_15'] = team_matches['Long_PPG_15'].fillna(1.35)

team_matches['Short_xG_5'] = team_matches['Short_xG_5'].fillna(1.35)
team_matches['Short_xGA_5'] = team_matches['Short_xGA_5'].fillna(1.35)
team_matches['Short_xGD_5'] = team_matches['Short_xGD_5'].fillna(0.0)
team_matches['Short_Form_5'] = team_matches['Short_Form_5'].fillna(6.5)
team_matches['Short_GF_5'] = team_matches['Short_GF_5'].fillna(1.35)
team_matches['Short_GA_5'] = team_matches['Short_GA_5'].fillna(1.35)
team_matches['Short_GD_5'] = team_matches['Short_GD_5'].fillna(0.0)
team_matches['Short_Net_ST_5'] = team_matches['Short_Net_ST_5'].fillna(0.0)

team_matches['Venue_PPG_8'] = team_matches['Venue_PPG_8'].fillna(1.35)
team_matches['Venue_GD_8'] = team_matches['Venue_GD_8'].fillna(0.0)
team_matches['Venue_xGD_8'] = team_matches['Venue_xGD_8'].fillna(0.0)

print("Dual-horizon xG and performance features calculated successfully!")

Dual-horizon xG and performance features calculated successfully!


### Step 7: Assemble Match-Level Dataset & Compute Relative xG Differentials
We merge home and away perspectives onto `Match_ID` and construct meaningful **relative matchup advantages** (Quality xGD differentials, Momentum xGD differentials, Net Shots on Target, and Venue records).

In [7]:
feature_cols = [
    'Match_ID', 'Team', 'Rest_Days',
    'Long_xG_15', 'Long_xGA_15', 'Long_xGD_15',
    'Long_GF_15', 'Long_GA_15', 'Long_GD_15', 'Long_Net_ST_15', 'Long_PPG_15',
    'Short_xG_5', 'Short_xGA_5', 'Short_xGD_5',
    'Short_Form_5', 'Short_GF_5', 'Short_GA_5', 'Short_GD_5', 'Short_Net_ST_5',
    'Venue_PPG_8', 'Venue_GD_8', 'Venue_xGD_8'
]

home_features = team_matches[team_matches['Is_Home'] == 1][feature_cols].copy()
home_features.columns = ['Match_ID', 'HomeTeam'] + [f'Home_{c}' for c in feature_cols[2:]]

away_features = team_matches[team_matches['Is_Home'] == 0][feature_cols].copy()
away_features.columns = ['Match_ID', 'AwayTeam'] + [f'Away_{c}' for c in feature_cols[2:]]

dataset = df_raw.merge(home_features, on=['Match_ID', 'HomeTeam'], how='left')
dataset = dataset.merge(away_features, on=['Match_ID', 'AwayTeam'], how='left')

# Relative Matchup Differentials (Actual xG & Traditional)
dataset['Diff_Quality_xGD'] = dataset['Home_Long_xGD_15'] - dataset['Away_Long_xGD_15']
dataset['Diff_Short_xGD'] = dataset['Home_Short_xGD_5'] - dataset['Away_Short_xGD_5']
dataset['Diff_Quality_xG'] = dataset['Home_Long_xG_15'] - dataset['Away_Long_xG_15']
dataset['Diff_Venue_xGD'] = dataset['Home_Venue_xGD_8'] - dataset['Away_Venue_xGD_8']

dataset['Diff_Quality_GD'] = dataset['Home_Long_GD_15'] - dataset['Away_Long_GD_15']
dataset['Diff_Quality_Net_ST'] = dataset['Home_Long_Net_ST_15'] - dataset['Away_Long_Net_ST_15']
dataset['Diff_Quality_PPG'] = dataset['Home_Long_PPG_15'] - dataset['Away_Long_PPG_15']
dataset['Diff_Short_Form'] = dataset['Home_Short_Form_5'] - dataset['Away_Short_Form_5']
dataset['Diff_Short_GD'] = dataset['Home_Short_GD_5'] - dataset['Away_Short_GD_5']
dataset['Diff_Short_Net_ST'] = dataset['Home_Short_Net_ST_5'] - dataset['Away_Short_Net_ST_5']
dataset['Diff_Venue_PPG'] = dataset['Home_Venue_PPG_8'] - dataset['Away_Venue_PPG_8']
dataset['Diff_Rest_Days'] = dataset['Home_Rest_Days'] - dataset['Away_Rest_Days']

# Target: H:0, D:1, A:2
dataset['Target'] = dataset['FTR'].map({'H': 0, 'D': 1, 'A': 2})

output_file = (Path('dataset') if Path('dataset').exists() else Path('../dataset')) / 'processed_match_dataset.csv'
dataset.to_csv(output_file, index=False)
print(f"Final engineered dataset saved to {output_file} ({len(dataset)} matches)")
dataset[['Date', 'HomeTeam', 'AwayTeam', 'Elo_Diff', 'Home_xG', 'Away_xG', 'Diff_Quality_xGD', 'Diff_Short_xGD', 'FTR', 'Target']].tail()

Final engineered dataset saved to dataset/processed_match_dataset.csv (1929 matches)


,Date,HomeTeam,AwayTeam,Elo_Diff,Home_xG,Away_xG,Diff_Quality_xGD,Diff_Short_xGD,FTR,Target
1924,2026-09-19,Everton,Ipswich,231.212281,2.070,1.49,0.869000,-0.19300,H,0
1925,2026-09-20,Leeds United FC,Crystal Palace,52.909537,2.070,1.49,0.352714,0.09850,D,1
1926,2026-09-20,Man City,Sunderland AFC,333.927794,2.865,2.02,1.762690,0.97025,H,0
1927,2026-09-20,Bournemouth,Liverpool,-31.472307,2.070,1.49,0.132667,-0.11200,A,2
1928,2026-09-20,Fulham,Man United,-125.971330,2.070,1.49,-0.419333,-0.15900,D,1
